In [37]:
import os

import scanpy as sc
import scipy.sparse as sp

## Add condition to AnnData objects (if not done during harmonization)

In [2]:
load_folder_path = '/nfs/team361/sb75/DATASETS/silver/mmb0-P1-239b_1p'

In [39]:
load_folder_paths = [
    '/nfs/team361/sb75/DATASETS/silver/mmb0-P1-239b_1p',
    '/nfs/team361/sb75/DATASETS/silver/mmb0-P2-239b_1p',
    '/nfs/team361/sb75/DATASETS/silver/mmb0-P3-239b_1p',
    '/nfs/team361/sb75/DATASETS/silver/mmb0-P4-239b_1p',
    '/nfs/team361/sb75/DATASETS/silver/mmb0-P5-239b_1p',
    '/nfs/team361/sb75/DATASETS/silver/smb1-20b_1p'
]

n_cells = 0

for load_folder_path in load_folder_paths:
    for subdir, _, files in os.walk(load_folder_path):
        for file in files:
            if file.endswith('.h5ad'):
                file_path = os.path.join(subdir, file)
                adata = sc.read_h5ad(file_path)
                adata.X = sp.csr_matrix(adata.X)
                adata.uns['condition'] = 'healthy'
                adata.write(file_path)
                n_cells += len(adata)

print(f"Total number of cells: {n_cells}")

Total number of cells: 9447075


In [38]:
load_folder_paths = [
    #'/nfs/team361/sb75/DATASETS/silver/mmb0-P1-239b_1p',
    #'/nfs/team361/sb75/DATASETS/silver/mmb0-P2-239b_1p',
    '/nfs/team361/sb75/DATASETS/silver/mmb0-P3-239b_1p',
    #'/nfs/team361/sb75/DATASETS/silver/mmb0-P4-239b_1p',
    #'/nfs/team361/sb75/DATASETS/silver/mmb0-P5-239b_1p',
]

n_cells = 0

for load_folder_path in load_folder_paths:
    for subdir, _, files in os.walk(load_folder_path):
        for file in files:
            if file.endswith('.h5ad'):
                file_path = os.path.join(subdir, file)
                adata = sc.read_h5ad(file_path)
                print(sp.csr_matrix(adata.X))
                n_cells += len(adata)

print(f"Total number of cells: {n_cells}")

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 3646688 stored elements and shape (34686, 1085)>
  Coords	Values
  (0, 14)	1.0
  (0, 51)	2.0
  (0, 118)	1.0
  (0, 216)	1.0
  (0, 221)	1.0
  (0, 235)	1.0
  (0, 251)	1.0
  (0, 287)	1.0
  (0, 329)	1.0
  (0, 337)	1.0
  (0, 370)	2.0
  (0, 385)	1.0
  (0, 396)	1.0
  (0, 402)	1.0
  (0, 407)	1.0
  (0, 415)	3.0
  (0, 436)	1.0
  (0, 441)	1.0
  (0, 446)	3.0
  (0, 485)	1.0
  (0, 503)	1.0
  (0, 511)	1.0
  (0, 515)	1.0
  (0, 526)	1.0
  (0, 619)	1.0
  :	:
  (34685, 216)	2.0
  (34685, 242)	1.0
  (34685, 319)	1.0
  (34685, 337)	1.0
  (34685, 418)	2.0
  (34685, 455)	1.0
  (34685, 459)	1.0
  (34685, 481)	4.0
  (34685, 504)	1.0
  (34685, 511)	1.0
  (34685, 638)	5.0
  (34685, 645)	1.0
  (34685, 671)	1.0
  (34685, 729)	1.0
  (34685, 765)	1.0
  (34685, 800)	1.0
  (34685, 863)	1.0
  (34685, 882)	1.0
  (34685, 891)	1.0
  (34685, 910)	1.0
  (34685, 915)	1.0
  (34685, 949)	1.0
  (34685, 960)	2.0
  (34685, 1030)	1.0
  (34685, 1037)	1.0
<Compressed Spars

KeyboardInterrupt: 

In [ ]:
gene_count_dict = {}

for i, gene in enumerate(adata.var['ensembl_id']):
    gene_count_dict[gene].append(adata.X[:, i].toarray())

In [19]:
import numpy as np
from collections import defaultdict

In [21]:
gene_count_dict = defaultdict(lambda: np.empty((0,)))

load_folder_paths = [
    '/nfs/team361/sb75/DATASETS/silver/mmb0-P1-239b_1p',
]

n_cells = 0

for load_folder_path in load_folder_paths:
    for subdir, _, files in os.walk(load_folder_path):
        for file in files:
            if file.endswith('.h5ad'):
                file_path = os.path.join(subdir, file)
                adata = sc.read_h5ad(file_path)
                n_cells += len(adata)
                for i, gene in enumerate(adata.var['ensembl_id']):
                    values = adata.X[:, i].toarray().flatten()
                    gene_count_dict[gene] = np.concatenate([gene_count_dict[gene], values])                
            print(f"Running number of cells: {n_cells}")

Running number of cells: 1304008


In [31]:
all_stats_dict = {}

for gene in gene_count_dict.keys():
    all_stats_dict[gene] = {}
    median = np.median(gene_count_dict[gene])
    mean = np.mean(gene_count_dict[gene])
    nz_values = gene_count_dict[gene][gene_count_dict[gene] != 0]
    if nz_values.size > 0:
        nz_median = np.median(nz_values)
        nz_mean = np.mean(nz_values)
    else:
        nz_median = np.nan
        nz_mean = np.nan

    all_stats_dict[gene]['median'] = median
    all_stats_dict[gene]['mean'] = mean
    all_stats_dict[gene]['nonzero_median'] = nz_median
    all_stats_dict[gene]['nonzero_mean'] = nz_mean

In [56]:
import pandas as pd
all_stats_df = pd.DataFrame(all_stats_dict).T

In [64]:
adata

AnnData object with n_obs × n_vars = 15791 × 1015
    obs: 'Main_molecular_cell_type', 'Sub_molecular_cell_type', 'Main_molecular_tissue_region', 'Sub_molecular_tissue_region', 'Molecular_spatial_cell_type', 'dataset', 'cell_type', 'cell_type_ontology_term_id', 'n_genes', 'cell_id'
    var: 'ensembl_id'
    uns: 'age_group', 'age_num', 'age_unit', 'assay', 'batch', 'condition', 'dataset_id', 'disease_level1', 'disease_level2', 'disease_level3', 'log1p', 'sample', 'sex', 'species', 'study', 'tissue'
    obsm: 'spatial'

In [65]:
adata.X.sum(axis=1).reshape(-1, 1)

matrix([[ 54.],
        [217.],
        [235.],
        ...,
        [ 67.],
        [ 56.],
        [185.]], shape=(15791, 1), dtype=float32)

In [67]:
basis_target_read_depth = -752.364435146444
target_read_depth_per_gene = 0.8921817095038856

In [69]:
# Perform gene corrected read depth normalization
test = adata.X / adata.X.sum(axis=1).reshape(-1, 1) * (
    basis_target_read_depth + adata.X.shape[1] *
    target_read_depth_per_gene)

In [71]:
test.toarray()

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 4.573134  , 2.286567  ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.82810813, ..., 0.        , 0.        ,
        0.        ]], shape=(15791, 1015), dtype=float32)

In [72]:
adata.X.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 2., 1., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.]], shape=(15791, 1015), dtype=float32)

In [63]:
all_stats_df

,gene_id,median,mean,nonzero_median,nonzero_mean
0,ENSMUSG00000024798,0.0,0.134563,1.0,1.539057
1,ENSMUSG00000042385,0.0,0.004279,1.0,1.010138
2,ENSMUSG00000036198,0.0,0.060855,1.0,2.016620
3,ENSMUSG00000028780,0.0,0.238804,1.0,2.072531
4,ENSMUSG00000015843,0.0,0.059675,1.0,1.287807
...,...,...,...,...,...
1080,ENSMUSG00000015619,0.0,0.055619,1.0,1.600388
1081,ENSMUSG00000070047,0.0,0.609236,1.0,1.893610
1082,ENSMUSG00000002266,0.0,0.077470,1.0,1.428262
1083,ENSMUSG00000036111,0.0,0.109547,1.0,1.387661
